# Thiết lập Dữ liệu - Máy 2 (Chi nhánh Nguyễn Huệ - CN002)

File này chạy độc lập trên Máy 2 để tự lấy dữ liệu của CN002 về từ `fahasa_dataset.py`.

## 1. Kết nối MongoDB Máy 2

In [3]:
import pymongo
import sys
import os

IP_MAY_2 = '100.118.157.52' 

try:
    client = pymongo.MongoClient(f'mongodb://{IP_MAY_2}:27017/', serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    print('-> Đã kết nối Máy 2 thành công!')
except Exception as e:
    print('LỖI KẾT NỐI. HÃY KIỂM TRA LẠI IP HOẶC TƯỜNG LỬA:', e)

-> Đã kết nối Máy 2 thành công!


## 2. Làm sạch Database

In [4]:
db = client['fahasa_btl2']

for collection in db.list_collection_names():
    db[collection].drop()
    
print('Đã làm sạch database trên Máy 2.')

Đã làm sạch database trên Máy 2.


## 3. Lọc và Insert Dữ liệu cho CN002

- Bảng dùng chung: Sách, Khách hàng, Chi nhánh.
- Bảng phân mảnh (chỉ lấy CN002): Nhân viên, Kho nhập, Kho bán, Hóa đơn.

In [5]:
# Import dữ liệu
try:
    sys.path.append(os.path.dirname(os.getcwd()))
    from fahasa_dataset import (
        chinhanh_list, sach_list, nhanvien_list, 
        khachhang_list, kho_nhap_list, kho_ban_list, hoadon_list
    )
except ImportError:
    sys.path.append('.')
    from fahasa_dataset import (
        chinhanh_list, sach_list, nhanvien_list, 
        khachhang_list, kho_nhap_list, kho_ban_list, hoadon_list
    )

# A. DỮ LIỆU NHÂN BẢN (REPLICATION)
db.chinhanh.insert_many(chinhanh_list)
db.sach.insert_many(sach_list)
db.khachhang.insert_many(khachhang_list)
print('Đã chèn bảng dùng chung (Sách, Chi nhánh, Khách hàng).')

# B. DỮ LIỆU PHÂN MẢNH (Chỉ lấy CN002)
def filter_cn2(data_list):
    return [item for item in data_list if item['MaCN'] == 'CN002']

nv_cn2 = filter_cn2(nhanvien_list)
kn_cn2 = filter_cn2(kho_nhap_list)
kb_cn2 = filter_cn2(kho_ban_list)
hd_cn2 = filter_cn2(hoadon_list)

db.nhanvien.insert_many(nv_cn2)
db.kho_nhap.insert_many(kn_cn2)
db.kho_ban.insert_many(kb_cn2)
print(f'Đã chèn {len(nv_cn2)} nhân viên, {len(kn_cn2)} kho nhập, {len(kb_cn2)} kho bán.')

# Insert Hoá Đơn theo chunk để tránh đầy RAM
chunk_size = 1000
for i in range(0, len(hd_cn2), chunk_size):
    db.hoadon.insert_many(hd_cn2[i:i+chunk_size])
    
print(f'Đã chèn tổng cộng {len(hd_cn2)} hóa đơn.')
print('HOÀN TẤT RÓT DỮ LIỆU CHO MÁY 2 (CN002)!')

Đã chèn bảng dùng chung (Sách, Chi nhánh, Khách hàng).
Đã chèn 30 nhân viên, 100 kho nhập, 100 kho bán.
Đã chèn tổng cộng 4978 hóa đơn.
HOÀN TẤT RÓT DỮ LIỆU CHO MÁY 2 (CN002)!
